# The `mpas` driver — reading a global model grid

**What the driver does.** **MPAS** (Model for Prediction Across Scales) is a weather/climate model whose grid is
a mesh of **Voronoi cells covering the whole globe** — mostly hexagons, plus exactly 12 pentagons. The `mpas`
driver reads it into a pyramids `MeshDataset`.

**The sample file.** `examples/data/real-data/mpas/x1.1024002.grid.nc` is a global grid of **~1 million cells**.
Watch out: MPAS stores coordinates in **radians**, not degrees.

## 1. Read the file

## Getting the data

The sample file(s) this notebook reads are **not stored in the repository** (they are large and carry per-source licensing). The cell below downloads just this driver's data from the project's public **Google Drive** folder ([link](https://drive.google.com/drive/folders/16IkuKUkCo21uDb_X1iZciSPp7X99bmjv)) into `examples/data/real-data/mpas/` — but only if it is not already there, so it is a no-op once you have the data. It uses the standard library only.

In [ ]:
# --- Getting the data (download it if you do not already have it) ------------
# The real-data samples are not committed to the repository (large, per-source
# licensing). If the file this notebook needs is not already under
# examples/data/real-data/, download it from the project's public Google Drive
# folder below. Standard library only.
#   https://drive.google.com/drive/folders/16IkuKUkCo21uDb_X1iZciSPp7X99bmjv
from pathlib import Path
import urllib.request, urllib.parse, http.cookiejar, re, html, zipfile

DATA = Path("../../../examples/data/real-data")

def _download_drive(file_id, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    op = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar()))
    op.addheaders = [("User-Agent", "Mozilla/5.0")]
    base = "https://drive.usercontent.google.com/download"
    r = op.open(base + "?" + urllib.parse.urlencode({"id": file_id, "export": "download", "confirm": "t"}))
    if r.headers.get_content_type() == "text/html":          # virus-scan page -> re-submit its form
        form = dict(re.findall(r'name="([^"]+)"\s+value="([^"]*)"', r.read().decode("utf-8", "replace")))
        r = op.open(base + "?" + urllib.parse.urlencode(form))
    with open(dest, "wb") as fh:
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            fh.write(chunk)

marker = DATA / "mpas" / "x1.1024002.grid.nc"
if not marker.exists():
    print("Downloading mpas sample (~1.4 GB) from Google Drive ...")
    _download_drive("1Bn0WMw2WVq9qIWiq7b-K2BL54GK8qWmy", marker)
    print("Saved:", marker)
else:
    print("mpas data already present:", marker)


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from cleopatra.config import Config

from pyramids.netcdf.ugrid import MeshDataset

DATA = Path("../../../examples/data/real-data")
grid = DATA / "mpas" / "x1.1024002.grid.nc"

t0 = time.perf_counter()
ds = MeshDataset.read_file(grid, driver="mpas")
secs = time.perf_counter() - t0
Config.set_matplotlib_backend()
print(f"read in {secs:.2f} s")
type(ds).__name__

## 2. Explore the mesh — attributes

The `bounds` run about `-3.14 … 3.14` by `-1.57 … 1.57`: those are **radians** (`-π … π` longitude,
`-π/2 … π/2` latitude), not degrees — the key thing to remember about MPAS.

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds_radians": tuple(round(b, 3) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

Over **a million faces** covering the whole planet, coordinates in radians, and **no data fields**
(`data_variable_names` is empty — this file is pure geometry, just the grid shape). So there is no height or
value to colour by; what we plot is the **mesh itself** (the cell edges).

## 3. How faces are built — the connectivity

MPAS cells are **Voronoi polygons**, so faces do not all have the same number of nodes. Counting the valid nodes
per face shows the mix — mostly hexagons (6 nodes), with the famous **12 pentagons** (5 nodes) globally.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
widths = (conn >= 0).sum(axis=1)
print("connectivity shape:", conn.shape)
pd.Series({int(k): int((widths == k).sum()) for k in np.unique(widths)},
          name="faces").rename_axis("nodes/face")

## 4. Coordinates — radians to degrees

Because MPAS is in radians, convert with `np.degrees(...)` to see the familiar lon/lat ranges and confirm the
grid really is global.

In [ ]:
lon = np.degrees(ds.mesh.node_x)
lat = np.degrees(ds.mesh.node_y)
print("longitude: %.0f ... %.0f degrees" % (lon.min(), lon.max()))
print("latitude:  %.0f ... %.0f degrees" % (lat.min(), lat.max()))

## 5. Plot the cells — `crop()` + `ds.plot_outline()`

A million cells is too many to draw at once, so we `crop(...)` a small region and draw its **cell edges** with
`ds.plot_outline()`. (We pick a region in **positive** longitude so the bbox does not cross longitude 0 — a crop
across 0 currently mis-selects the whole globe: pyramids-core#1591 / pyramids-mesh#119.)

In [ ]:
region = ds.crop(bbox=(0.80, 0.30, 0.95, 0.45))   # radians; positive longitude (no x=0 crossing)
print("cells in this region:", region.n_face)

glyph = region.plot_outline()
glyph.ax.set_aspect("equal")
glyph.ax.set_title("MPAS grid — a small region (cell outlines)")

**What you are looking at:** the outline of each grid cell in a small patch of the globe. **Each little
shape is one MPAS cell** — a **Voronoi hexagon** (globally, 12 of them are pentagons). Every cell is about the
same size: that even, quasi-uniform spacing everywhere on the planet is the whole point of an MPAS mesh.

## Takeaway

- The `mpas` driver reads a **million-cell** global grid into a pyramids `MeshDataset` in a couple of seconds.
- Explore it with the usual attributes and methods; the connectivity shows it is a **Voronoi** mesh (hexagons +
  12 pentagons), the coordinates are in **radians** (convert with `np.degrees(...)`), and it carries **no data
  fields** — so we plot the geometry.
- Plot with `crop()` + `ds.plot_outline()` to see the cell structure.

> **Note.** A regional `crop()` across longitude 0 currently mis-selects the whole globe
> (pyramids-core#1591 / pyramids-mesh#119); until fixed, crop regions that stay on one side of 0.